# Dashboard Interactivo — Pobreza Multidimensional Los Ríos

Dashboard construido con `ipywidgets` y `matplotlib` sobre datos CASEN 2017–2024.
Incluye análisis de Educación, Vivienda, Empleo y Composición del Hogar, más módulos de ML (Random Forest y Clustering Jerárquico).

## 📦 Importaciones y Carga de Modelos ML

In [1]:
import unicodedata
import seaborn as sns
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import ipywidgets as ipw
from IPython.display import display, clear_output
import warnings

import joblib

try:
    rf_idx          = joblib.load("modelos_ml/modelo_rf_idx.pkl")
    INDICES_CLUSTER = joblib.load("modelos_ml/indices_cluster.pkl")
    K               = joblib.load("modelos_ml/k_clusters.pkl")
    df_cl = df_idx  = pd.read_csv("modelos_ml/df_idx.csv")
    # Reconstruir cluster desde los índices si no está en el CSV
    if "cluster" not in df_cl.columns:
        from scipy.cluster.hierarchy import linkage, fcluster
        from sklearn.preprocessing import StandardScaler
        _X = df_cl[INDICES_CLUSTER].dropna()
        _Xs = StandardScaler().fit_transform(_X)
        _Z  = linkage(_Xs, method="ward")
        _lbl = fcluster(_Z, K, criterion="maxclust")
        df_cl["cluster"] = np.nan
        df_cl.loc[_X.index, "cluster"] = _lbl
        print(f"   ⚙️  cluster reconstruido al vuelo (K={K})")
    labels = df_cl["cluster"].dropna().astype(int).values
    ML_OK = True
    print(f"✅ Modelos cargados | K={K} | Índices: {INDICES_CLUSTER}")
except FileNotFoundError as e:
    ML_OK = False
    print(f"⚠️  Archivo no encontrado: {e}")
    print("   Ejecuta random_forest_losrios.ipynb y corre la celda de guardado primero.")
warnings.filterwarnings('ignore')

   ⚙️  cluster reconstruido al vuelo (K=4)
✅ Modelos cargados | K=4 | Índices: ['idx_educacion', 'idx_vivienda', 'idx_composicion', 'idx_ingresos']


## ⚙️ Configuración y Constantes

In [2]:
# Configuración y constantes
 
ML_DIR = Path("modelos_ml")  # Carpeta con modelos ML
CSV_PATH = Path("Educación") / "casen_losrios_2017_2024.csv"
SHAPEFILE = Path("Vivienda") / "crawler" / "comunas" / "comunas.shp"
INDICADORES_CSV = Path("Vivienda") / "indicadores_vivienda.csv"
INDICADOR_MAPA = "carencia_servicios_basicos"
AÑOS_MAPA = [2017, 2024]
 
CODIGO_COMUNA = {
    '14101': 'Valdivia',
    '14102': 'Corral',
    '14103': 'Lanco',
    '14104': 'Los Lagos',
    '14105': 'Máfil',
    '14106': 'Mariquina',
    '14107': 'Paillaco',
    '14108': 'Panguipulli',
    '14201': 'La Unión',
    '14202': 'Futrono',
    '14203': 'Lago Ranco',
    '14204': 'Río Bueno',
}
 
NIVEL_LABEL = {
    1: 'Sin ed. formal',
    2: 'Básica',
    3: 'Básica',
    4: 'Básica',
    5: 'Básica',
    6: 'Media',
    7: 'Media',
    8: 'Media',
    9: 'Superior',
    10: 'Superior',
    11: 'Superior',
    12: 'Superior',
    13: 'Superior',
    14: 'Sin info',
    15: 'Sin info',
}
NIVELES_ORDEN = ['Sin ed. formal', 'Básica', 'Media', 'Superior']
 
C17 = '#185FA5'
C24 = '#EF9F27'
COK = '#1D9E75'
CWRN = '#E24B4A'
CBG = '#F9F9F7'

## 📂 Carga y Procesamiento de Datos

In [3]:
# Carga y procesamiento de datos
 
def cargar_datos(path):
    df = pd.read_csv(path)
    df['cod_comuna'] = df['estrato'].astype(str).str[:5]
    df['comuna'] = df['cod_comuna'].map(CODIGO_COMUNA)
    df['nivel_grupo'] = df['nivel_educacion'].map(NIVEL_LABEL)
    df['analfabeto'] = (df['alfabetismo'] != 1).astype(float)
    df['desertor'] = (
        df['edad'].between(6, 24) &
        (df['asiste_actualmente'] == 2) &
        df['nivel_educacion'].notna() &
        (df['nivel_educacion'] > 1)
    ).astype(float)
    return df
 
try:
    DF = cargar_datos(CSV_PATH)
except FileNotFoundError:
    raise FileNotFoundError(
        f"No se encontró '{CSV_PATH}'. Asegúrate de ejecutar este notebook desde la raíz del repo."
    )
 
COMUNAS_ALL = sorted(DF['comuna'].dropna().unique())
AÑOS = sorted(DF['año'].unique())
 
 
def normalizar_texto(valor):
    texto = unicodedata.normalize('NFKD', str(valor))
    texto = ''.join(c for c in texto if not unicodedata.combining(c))
    return ' '.join(texto.lower().split())
 
def cargar_mapas():
    gdf = gpd.read_file(SHAPEFILE)
    gdf = gdf[gdf['codregion'].astype(str) == '14'].copy()
    gdf['key'] = gdf['Comuna'].map(normalizar_texto)
 
    indicadores = pd.read_csv(INDICADORES_CSV, encoding='utf-8-sig')
    mapas = {}
    for año in AÑOS_MAPA:
        col = f'{INDICADOR_MAPA}_{año}'
        datos = indicadores[['comuna', col]].rename(columns={col: 'valor'})
        datos['key'] = datos['comuna'].map(normalizar_texto)
        mapas[año] = gdf.merge(datos, on='key', how='left')
 
    diferencia = mapas[AÑOS_MAPA[0]][['key', 'Comuna', 'geometry', 'valor']].rename(columns={'valor': 'valor_inicial'})
    diferencia = diferencia.merge(
        mapas[AÑOS_MAPA[1]][['key', 'valor']].rename(columns={'valor': 'valor_final'}),
        on='key',
    )
    diferencia['valor'] = diferencia['valor_final'] - diferencia['valor_inicial']
    return mapas, diferencia
 
try:
    MAPAS, DIFERENCIA_MAPA = cargar_mapas()
    LIMITE_DIFERENCIA = DIFERENCIA_MAPA['valor'].abs().max()
    MAPAS_OK = True
except Exception:
    MAPAS, DIFERENCIA_MAPA, LIMITE_DIFERENCIA, MAPAS_OK = {}, None, 0, False

## 📊 Cálculo de Métricas

In [4]:
# Cálculos métricas
 
def subconjunto(df, comunas, año):
    return df[df['comuna'].isin(comunas) & (df['año'] == año)]
 
def porcentaje(parte, base):
    return round(len(parte) / len(base) * 100, 1) if len(base) > 0 else 0
 
def sufijo_comunas(comunas):
    return f' ({", ".join(comunas)})' if len(comunas) <= 3 else f' ({len(comunas)} comunas)'
 
 
def tasa_desercion(df, comunas, año):
    sub = subconjunto(df, comunas, año)
    jovenes = sub[sub['edad'].between(6, 24)]
    return porcentaje(jovenes[jovenes['desertor'] == 1], jovenes)
 
def tasa_analfabetismo_jefes(df, comunas, año):
    sub = subconjunto(df, comunas, año)
    jefes = sub[sub['jefe_hogar'] == 1]
    return porcentaje(jefes[jefes['analfabeto'] == 1], jefes)
 
def tasa_exclusion_laboral(df, comunas, año):
    sub = subconjunto(df, comunas, año)
    jovenes = sub[sub['edad'].between(15, 29)]
    return porcentaje(jovenes[jovenes['desertor'] == 1], jovenes)
 
def tasa_sin_media_adultos(df, comunas, año):
    sub = subconjunto(df, comunas, año)
    adultos = sub[sub['edad'].between(15, 64)]
    return porcentaje(adultos[adultos['nivel_grupo'].isin(['Sin ed. formal', 'Básica'])], adultos)
 
def tasa_adultos_mayores(df, comunas, año):
    sub = subconjunto(df, comunas, año)
    return porcentaje(sub[sub['edad'] >= 65], sub)
 
def tasa_ninos(df, comunas, año):
    sub = subconjunto(df, comunas, año)
    return porcentaje(sub[sub['edad'] < 15], sub)
 
 
def kpi_educacion(df, comunas, año):
    sub = subconjunto(df, comunas, año)
    adultos = sub[sub['edad'].between(15, 64)]
    kpi1 = porcentaje(adultos[adultos['nivel_grupo'].isin(['Sin ed. formal', 'Básica', 'Media'])], adultos)
    kpi2 = tasa_analfabetismo_jefes(df, comunas, año)
    return kpi1, kpi2
 
def vol_educacion(df, comunas, año):
    sub = subconjunto(df, comunas, año)
    total = len(sub)
    adultos = int(sub['edad'].between(15, 64).sum())
    return total, adultos
 
def tabla_niveles(df, comunas, año):
    sub = subconjunto(df, comunas, año)
    jefes = sub[sub['jefe_hogar'] == 1]
    counts = jefes['nivel_grupo'].value_counts()
    total = counts.sum()
    return {nv: round(counts.get(nv, 0) / total * 100, 1) for nv in NIVELES_ORDEN}
 
 
def aplicar_fA(personas, dormitorios):
    if dormitorios <= 0:
        return None, None, None
    A = round(personas / dormitorios, 2)
    if A < 2.5:
        return 1, 'Sin hacinamiento', A
    elif A < 3.5:
        return 2, 'Hacinamiento medio', A
    elif A < 5.0:
        return 3, 'Hacinamiento alto', A
    else:
        return 4, 'Hacinamiento crítico', A
 
 
def kpi_vivienda(df, comunas, año):
    if not MAPAS_OK:
        return 0, 0
    claves = [normalizar_texto(c) for c in comunas]
    mapa_año = MAPAS.get(año)
    carencia = mapa_año[mapa_año['key'].isin(claves)]['valor'] if mapa_año is not None else pd.Series(dtype=float)
    cambio = DIFERENCIA_MAPA[DIFERENCIA_MAPA['key'].isin(claves)]['valor']
    kpi1 = round(carencia.mean(), 1) if len(carencia) else 0
    kpi2 = round(cambio.mean(), 1) if len(cambio) else 0
    return kpi1, kpi2
 
def vol_vivienda(df, comunas, año):
    sub = subconjunto(df, comunas, año)
    hogares = sub['id_hogar'].nunique()
    personas_por_hogar = round(len(sub) / hogares, 1) if hogares else 0
    return hogares, personas_por_hogar
 
 
def kpi_empleo(df, comunas, año):
    return tasa_exclusion_laboral(df, comunas, año), tasa_sin_media_adultos(df, comunas, año)
 
def vol_empleo(df, comunas, año):
    sub = subconjunto(df, comunas, año)
    return int(sub['edad'].between(15, 64).sum()), int(len(sub))
 
 
def kpi_composicion(df, comunas, año):
    sub = subconjunto(df, comunas, año)
    jefes = sub[sub['jefe_hogar'] == 1]
    kpi1 = porcentaje(jefes[jefes['analfabeto'] == 1], jefes)
    return kpi1, tasa_adultos_mayores(df, comunas, año)
 
def vol_composicion(df, comunas, año):
    sub = subconjunto(df, comunas, año)
    return sub['id_hogar'].nunique(), int(len(sub))

## 📈 Funciones de Graficado

In [5]:
# Funciones de graficado
 
def estilo_ax(ax, titulo='', xlabel='', ylabel=''):
    ax.set_facecolor('white')
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_color('#DDDDDD')
    ax.tick_params(colors='#555', labelsize=9)
    if titulo:
        ax.set_title(titulo, fontsize=10, color='#222', pad=8, fontweight='normal')
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=9, color='#666')
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=9, color='#666')
 
 
def plot_barras_educacion(ax, comunas, sufijo=''):
    """Barras comparadas 2017 y 2024 lado a lado."""
    x = np.arange(len(NIVELES_ORDEN))
    w = 0.35
    for j, (año, color) in enumerate([(2017, C17), (2024, C24)]):
        vals = [tabla_niveles(DF, comunas, año).get(n, 0) for n in NIVELES_ORDEN]
        bars = ax.bar(x + (j - 0.5) * w, vals, w, color=color, label=str(año), zorder=3, alpha=0.9)
        for bar in bars:
            h = bar.get_height()
            if h > 0:
                ax.text(bar.get_x() + bar.get_width() / 2, h + 0.5,
                        f'{h:.1f}%', ha='center', va='bottom', fontsize=7.5, color='#444')
    ax.set_xticks(x)
    ax.set_xticklabels(NIVELES_ORDEN, fontsize=9)
    ax.yaxis.grid(True, color='#EEEEEE', zorder=0)
    ax.set_axisbelow(True)
    ax.legend(fontsize=8, framealpha=0)
    estilo_ax(ax, titulo=f'Nivel educacional, jefes de hogar{sufijo} (2017 / 2024)', ylabel='%')
 
 
def plot_barras_educacion_single(ax, comunas, año, sufijo=''):
    """Barras para un único año."""
    color = C17 if año == 2017 else C24
    x = np.arange(len(NIVELES_ORDEN))
    vals = [tabla_niveles(DF, comunas, año).get(n, 0) for n in NIVELES_ORDEN]
    bars = ax.bar(x, vals, 0.5, color=color, label=str(año), zorder=3, alpha=0.9)
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.5,
                    f'{h:.1f}%', ha='center', va='bottom', fontsize=7.5, color='#444')
    ax.set_xticks(x)
    ax.set_xticklabels(NIVELES_ORDEN, fontsize=9)
    ax.yaxis.grid(True, color='#EEEEEE', zorder=0)
    ax.set_axisbelow(True)
    ax.legend(fontsize=8, framealpha=0)
    estilo_ax(ax, titulo=f'Nivel educacional, jefes de hogar{sufijo} ({año})', ylabel='%')
 
 
def plot_hacinamiento(ax, personas, dormitorios):
    f, label, A = aplicar_fA(personas, dormitorios)
    if f is None:
        ax.text(0.5, 0.5, 'Dormitorios debe ser > 0', ha='center', va='center', transform=ax.transAxes)
        return None, None, None
 
    cats = ['Sin hacinamiento\nf = 1', 'Hacinamiento medio\nf = 2',
            'Hacinamiento alto\nf = 3', 'Hacinamiento crítico\nf = 4']
    rangos = ['< 2,5 p/dorm', '2,5 – 3,5', '3,5 – 5,0', '≥ 5,0']
    colores = [COK, C24, '#D85A30', CWRN]
    anchos = [2.5, 1.0, 1.5, 2.0]
    alphas = [1.0 if i + 1 == f else 0.22 for i in range(4)]
 
    bars = ax.barh(cats, anchos, color=colores, height=0.52, zorder=3)
    for bar, a in zip(bars, alphas):
        bar.set_alpha(a)
    ax.set_xlim(0, 8)
    for i, (bar, rango) in enumerate(zip(bars, rangos)):
        c = '#222' if alphas[i] == 1.0 else '#999'
        ax.text(bar.get_width() + 0.15, bar.get_y() + bar.get_height() / 2,
                rango, va='center', fontsize=8.5, color=c)
 
    ax.set_ylim(-0.9, 3.5)
    pos = min(A, 7.5)
    ax.axvline(pos, color='#222', lw=1.8, ls='--', zorder=5)
    ax.text(pos, -0.7, f'A = {A}', ha='center', va='center', fontsize=8.5, color='#222',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#CCC', lw=0.8))
 
    ax.xaxis.set_visible(False)
    ax.spines[['top', 'right', 'bottom']].set_visible(False)
    ax.spines['left'].set_color('#DDD')
    estilo_ax(ax, titulo=f'Función de hacinamiento f(A): {personas} personas / {dormitorios} dorm.')
    return f, label, A
 
 
def plot_horizontal_comunas(ax, comunas, año, col_fn, titulo, xlabel):
    if año == '2017/2024':
        # Barras agrupadas por comuna: 2017 arriba, 2024 abajo
        registros_17 = {c: col_fn(DF, [c], 2017) for c in comunas}
        registros_24 = {c: col_fn(DF, [c], 2024) for c in comunas}
        # ordenar por valor 2017 descendente
        comunas_ord = sorted(comunas, key=lambda c: registros_17[c], reverse=True)

        y = np.arange(len(comunas_ord))
        h = 0.35
        bars17 = ax.barh(y + h/2, [registros_17[c] for c in comunas_ord],
                         h, color=C17, label='2017', zorder=3, alpha=0.9)
        bars24 = ax.barh(y - h/2, [registros_24[c] for c in comunas_ord],
                         h, color=C24, label='2024', zorder=3, alpha=0.9)

        for bar, c in zip(bars17, comunas_ord):
            v = registros_17[c]
            ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                    f'{v:.1f}%', va='center', fontsize=7.5, color='#444')
        for bar, c in zip(bars24, comunas_ord):
            v = registros_24[c]
            ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                    f'{v:.1f}%', va='center', fontsize=7.5, color='#444')

        ax.set_yticks(y)
        ax.set_yticklabels(comunas_ord, fontsize=9)
        ax.xaxis.grid(True, color='#EEEEEE', zorder=0)
        ax.set_axisbelow(True)
        todos = list(registros_17.values()) + list(registros_24.values())
        ax.set_xlim(0, max(todos or [0]) + 10)
        ax.legend(fontsize=8, framealpha=0)
        estilo_ax(ax, titulo=titulo, xlabel=xlabel)
    else:
        registros = sorted(((c, col_fn(DF, [c], año)) for c in comunas), key=lambda r: r[1], reverse=True)
        labels = [r[0] for r in registros]
        vals = [r[1] for r in registros]
        color = C17 if año == 2017 else C24

        bars = ax.barh(labels, vals, color=color, height=0.55, zorder=3, alpha=0.9)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                    f'{v:.1f}%', va='center', fontsize=8, color='#444')
        ax.xaxis.grid(True, color='#EEEEEE', zorder=0)
        ax.set_axisbelow(True)
        ax.set_xlim(0, max(vals or [0]) + 8)
        estilo_ax(ax, titulo=titulo, xlabel=xlabel)
 
def plot_mapa_carencia(ax, mapa, comunas, cmap, escala, titulo, etiqueta_barra):
    mapa.plot(column='valor', cmap=cmap, edgecolor='black', linewidth=0.5, ax=ax,
              legend=True, legend_kwds={'label': etiqueta_barra, 'shrink': 0.7},
              missing_kwds={'color': 'lightgrey'}, **escala)
    claves = [normalizar_texto(c) for c in comunas]
    seleccion = mapa[mapa['key'].isin(claves)]
    if not seleccion.empty:
        seleccion.boundary.plot(ax=ax, edgecolor=COK, linewidth=2.2, zorder=4)
    ax.set_facecolor(CBG)
    ax.set_title(titulo, fontsize=10, color='#222')
    ax.set_axis_off()

## 🧩 Interfaz — Componentes Reutilizables

In [6]:
# Interfaz y componentes reutilizables
 
def html_kpi(etiqueta, valor, unidad, color_borde):
    return f"""
    <div style="background:#F4F6FA;border-radius:8px;padding:9px 14px;
                border-left:4px solid {color_borde};min-width:168px">
      <div style="font-size:11px;color:#777;margin-bottom:3px">{etiqueta}</div>
      <div style="font-size:20px;font-weight:500;color:#1A1A1A">
        {valor}<span style="font-size:12px;color:#999;margin-left:2px">{unidad}</span>
      </div>
    </div>"""
 
def html_vol(etiqueta, valor):
    return f"""
    <div style="background:#F0F0EE;border-radius:8px;padding:9px 14px;min-width:140px">
      <div style="font-size:11px;color:#888;margin-bottom:3px">{etiqueta}</div>
      <div style="font-size:18px;font-weight:500;color:#444">{valor:,}</div>
    </div>"""
 
 
def crear_tab(cfg):
    # Si el tab admite la opción 2017/2024, se agrega al dropdown de año
    if cfg.get('año_comparado'):
        opciones_año = [(str(a), a) for a in AÑOS] + [('2017/2024', '2017/2024')]
    else:
        opciones_año = [(str(a), a) for a in AÑOS]
 
    dd_año = ipw.Dropdown(options=opciones_año, value=AÑOS[0],
                          layout=ipw.Layout(width='110px'))
    dd_graf = ipw.Dropdown(options=cfg.get('graf_opciones', [('Vista principal', 'main')]),
                           layout=ipw.Layout(width='230px'))
 
    kpi1_w = ipw.HTML()
    kpi2_w = ipw.HTML()
    vol1_w = ipw.HTML()
    vol2_w = ipw.HTML()
    nota_w = ipw.HTML(f'<div style="font-size:10px;color:#bbb;margin-top:2px">Fuente: {cfg["fuente"]}</div>')
    out_fig = ipw.Output()
    out_result = ipw.HTML()
 
    def refrescar(comunas_sel, año_sel, graf_sel, extras_vals):
        # Para KPIs y volumen, si año es '2017/2024' usar el primero disponible
        año_kpi = AÑOS[0] if año_sel == '2017/2024' else año_sel
        v1, v2 = cfg['kpi_fn'](DF, comunas_sel, año_kpi)
        n1, n2 = cfg['vol_fn'](DF, comunas_sel, año_kpi)
        kpi1_w.value = html_kpi(cfg['kpi_labels'][0], v1, cfg['kpi_unidades'][0], cfg['kpi_colores'][0])
        kpi2_w.value = html_kpi(cfg['kpi_labels'][1], v2, cfg['kpi_unidades'][1], cfg['kpi_colores'][1])
        vol1_w.value = html_vol(cfg['vol_labels'][0], n1)
        vol2_w.value = html_vol(cfg['vol_labels'][1], n2)
 
        out_result.value = ""
 
        with out_fig:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(7.2, 3.8))
            fig.patch.set_facecolor(CBG)
            ax.set_facecolor('white')
            resultado = cfg['render_fn'](ax, comunas_sel, año_sel, graf_sel, extras_vals)
            plt.tight_layout(pad=1.2)
            plt.show()
            if resultado is not None:
                out_result.value = resultado
 
    encabezado = ipw.HTML(f"""
    <div style="padding:6px 0 8px;border-bottom:1.5px solid #E8E8E8;margin-bottom:10px">
      <span style="font-size:15px;font-weight:500;color:#1A1A1A">{cfg['nombre']}</span>
    </div>""")
    lbl_kpis = ipw.HTML('<div style="font-size:11px;color:#999;margin-bottom:5px">KPIs del estudio</div>')
    lbl_vol = ipw.HTML('<div style="font-size:11px;color:#999;margin-bottom:5px">Volumen de muestra</div>')
    fila_kpi = ipw.HBox(
        [ipw.VBox([lbl_kpis, ipw.HBox([kpi1_w, kpi2_w], layout=ipw.Layout(gap='8px'))]),
         ipw.VBox([lbl_vol,  ipw.HBox([vol1_w, vol2_w], layout=ipw.Layout(gap='8px'))])],
        layout=ipw.Layout(gap='20px', align_items='flex-start', margin='0 0 12px 0')
    )
    bloque_año = ipw.HBox(
        [ipw.HTML('<span style="font-size:11px;color:#999;line-height:28px;margin-left:10px">Año:</span>'), dd_año],
        layout=ipw.Layout(align_items='center', gap='5px')
    )
    controles = ipw.HBox(
        [ipw.HTML('<span style="font-size:11px;color:#999;line-height:28px">Gráfico:</span>'), dd_graf, bloque_año],
        layout=ipw.Layout(align_items='center', gap='5px', margin='0 0 8px 0')
    )
    extras_box = ipw.VBox(cfg.get('extras_widgets', []), layout=ipw.Layout(margin='0 0 6px 0'))
    contenido = ipw.VBox([encabezado, fila_kpi, nota_w, controles, extras_box, out_fig, out_result],
                         layout=ipw.Layout(padding='14px'))
 
    graf_con_extras = cfg.get('graf_con_extras', set())
    graf_sin_año = cfg.get('graf_sin_año', set())
 
    def alternar_controles(*_):
        extras_box.layout.display = '' if dd_graf.value in graf_con_extras else 'none'
        bloque_año.layout.display = 'none' if dd_graf.value in graf_sin_año else ''
 
    dd_graf.observe(alternar_controles, names='value')
    alternar_controles()
 
    return contenido, refrescar, dd_año, dd_graf

## 🎓 Tab: Educación

In [7]:
# Educación
 
def render_educacion(ax, comunas, año, graf, extras):
    sufijo = sufijo_comunas(comunas)
    if graf == 'barras':
        if año == '2017/2024':
            plot_barras_educacion(ax, comunas, sufijo)
        else:
            plot_barras_educacion_single(ax, comunas, año, sufijo)
        return
    año_val = AÑOS[0] if año == '2017/2024' else año
    titulo_año = '2017 / 2024' if año == '2017/2024' else str(año)
    opciones = {
        'desercion':     (tasa_desercion,           f'Deserción escolar 6–24 años{sufijo}  ({titulo_año})', '%'),
        'analfabetismo': (tasa_analfabetismo_jefes, f'Analfabetismo jefes de hogar{sufijo}  ({titulo_año})', '%'),
        'edu_adultos':   (tasa_sin_media_adultos,   f'Adultos 15–64 sin ed. media{sufijo}  ({titulo_año})', '%'),
    }
    fn, titulo, xlabel = opciones[graf]
    plot_horizontal_comunas(ax, comunas, año, col_fn=fn, titulo=titulo, xlabel=xlabel)
 
cfg_edu = dict(
    nombre='Educación',
    kpi_fn=kpi_educacion,
    kpi_labels=['Adultos 15–64 sin ed. superior', 'Analfabetismo jefes de hogar'],
    kpi_unidades=['%', '%'],
    kpi_colores=[C17, CWRN],
    vol_fn=vol_educacion,
    vol_labels=['Total personas en muestra', 'Adultos 15–64'],
    graf_opciones=[
        ('Nivel educacional del jefe de hogar',      'barras'),
        ('Deserción escolar por comuna',      'desercion'),
        ('Analfabetismo jefes de hogar por comuna',    'analfabetismo'),
        ('Adultos sin ed. media por comuna',  'edu_adultos'),
    ],
    año_comparado=True,   # habilita la opción 2017/2024 en el dropdown de año
    render_fn=render_educacion,
    extras_widgets=[],
    fuente='CASEN 2017 y 2024, Región de Los Ríos'
)

## 🏠 Tab: Vivienda y Habitabilidad

In [8]:
# Vivienda
 
sl_personas = ipw.IntSlider(min=1, max=15, value=4, step=1, description='Personas:',
                            continuous_update=False, style={'description_width': '75px'},
                            layout=ipw.Layout(width='310px'))
sl_dorm = ipw.IntSlider(min=1, max=8, value=2, step=1, description='Dorm.:',
                        continuous_update=False, style={'description_width': '75px'},
                        layout=ipw.Layout(width='270px'))
 
COLORES_FA = {1: COK, 2: C24, 3: '#D85A30', 4: CWRN}
 
def resultado_hacinamiento(f, label, A):
    if f is None:
        return None
    c = COLORES_FA[f]
    return (f'<div style="margin-top:5px;padding:8px 14px;background:#F8F8F6;'
            f'border-radius:8px;border-left:4px solid {c}">'
            f'<span style="font-size:13px;color:#444">'
            f'Índice A = <b>{A}</b> personas/dormitorio, '
            f'f(A) = <b style="color:{c}">{f} ({label})</b></span></div>')
 
def render_vivienda(ax, comunas, año, graf, extras):
    if graf == 'fA':
        personas, dormitorios = extras
        f, label, A = plot_hacinamiento(ax, personas, dormitorios)
        return resultado_hacinamiento(f, label, A)
 
    if not MAPAS_OK:
        ax.text(0.5, 0.5, 'No se encontró el shapefile de comunas o indicadores_vivienda.csv\n'
                          '(deben estar en Vivienda/crawler/comunas/ y Vivienda/ del repo)',
                ha='center', va='center', transform=ax.transAxes, fontsize=9, color='#999')
        ax.set_axis_off()
        return None
 
    if graf == 'mapa_carencia':
        plot_mapa_carencia(ax, MAPAS[año], comunas, 'YlOrRd', {},
                           f'Carencia de servicios básicos por comuna ({año})', 'Porcentaje (%)')
    elif graf == 'mapa_diferencia':
        plot_mapa_carencia(ax, DIFERENCIA_MAPA, comunas, 'RdBu_r',
                           {'vmin': -LIMITE_DIFERENCIA, 'vmax': LIMITE_DIFERENCIA},
                           f'Cambio en la carencia de servicios básicos, {AÑOS_MAPA[0]} a {AÑOS_MAPA[1]}',
                           'Cambio en puntos porcentuales')
 
cfg_viv = dict(
    nombre='Vivienda y Habitabilidad',
    kpi_fn=kpi_vivienda,
    kpi_labels=['Carencia de servicios básicos', f'Cambio {AÑOS_MAPA[0]} a {AÑOS_MAPA[1]}'],
    kpi_unidades=['%', 'p.p.'],
    kpi_colores=[CWRN, C17],
    vol_fn=vol_vivienda,
    vol_labels=['Hogares únicos', 'Personas por hogar'],
    graf_opciones=[
        ('Simulador de hacinamiento f(A)', 'fA'),
        ('Mapa de carencia de servicios básicos', 'mapa_carencia'),
        ('Mapa de cambio entre 2017 y 2024', 'mapa_diferencia'),
    ],
    graf_con_extras={'fA'},
    graf_sin_año={'mapa_diferencia'},
    render_fn=render_vivienda,
    extras_widgets=[sl_personas, sl_dorm],
    fuente='CASEN 2017 y 2024 (hogares) / BCN (carencia de servicios básicos por comuna)'
)

## 💼 Tab: Empleo e Ingresos

In [9]:
# Empleo e Ingresos
 
def render_empleo(ax, comunas, año, graf, extras):
    sufijo = sufijo_comunas(comunas)
    opciones = {
        'excl_laboral': (tasa_exclusion_laboral,
                         f'Exclusión laboral temprana 15–29 años{sufijo}  ({año})',
                         '% jóvenes sin asistencia ni formación (proxy)'),
        'edu_adultos': (tasa_sin_media_adultos,
                        f'Adultos 15–64 sin educación media completa{sufijo}  ({año})', '%'),
    }
    fn, titulo, xlabel = opciones[graf]
    plot_horizontal_comunas(ax, comunas, año, col_fn=fn, titulo=titulo, xlabel=xlabel)
 
cfg_emp = dict(
    nombre='Empleo e Ingresos',
    kpi_fn=kpi_empleo,
    kpi_labels=['Excl. laboral 15–29 (proxy)', 'Sin media en edad laboral (proxy)'],
    kpi_unidades=['%', '%'],
    kpi_colores=[C17, '#D85A30'],
    vol_fn=vol_empleo,
    vol_labels=['Adultos 15–64', 'Total personas'],
    graf_opciones=[
        ('Exclusión laboral temprana por comuna', 'excl_laboral'),
        ('Adultos sin ed. media por comuna', 'edu_adultos'),
    ],
    render_fn=render_empleo,
    extras_widgets=[],
    fuente='CASEN 2017 y 2024 / ENE INE (proxies sobre columnas disponibles)'
)

## 👨‍👩‍👧 Tab: Composición del Hogar

In [10]:
# Composición del Hogar
 
def render_composicion(ax, comunas, año, graf, extras):
    sufijo = sufijo_comunas(comunas)
    opciones = {
        'analfabetismo': (tasa_analfabetismo_jefes,
                          f'Analfabetismo jefes de hogar{sufijo}  ({año})', '% jefes analfabetos'),
        'adultos_mayores': (tasa_adultos_mayores,
                            f'Proporción adultos mayores ≥65{sufijo}  ({año})', '% sobre total de personas'),
        'ninos': (tasa_ninos,
                  f'Proporción de niños <15 años{sufijo}  ({año})', '% sobre total de personas'),
    }
    fn, titulo, xlabel = opciones[graf]
    plot_horizontal_comunas(ax, comunas, año, col_fn=fn, titulo=titulo, xlabel=xlabel)
 
cfg_comp = dict(
    nombre='Composición del Hogar',
    kpi_fn=kpi_composicion,
    kpi_labels=['Analfabetismo jefes de hogar', 'Adultos mayores ≥65 en muestra'],
    kpi_unidades=['%', '%'],
    kpi_colores=[CWRN, COK],
    vol_fn=vol_composicion,
    vol_labels=['Hogares únicos', 'Total personas'],
    graf_opciones=[
        ('Analfabetismo jefes por comuna', 'analfabetismo'),
        ('Proporción adultos mayores por comuna', 'adultos_mayores'),
        ('Proporción niños <15 años por comuna', 'ninos'),
    ],
    render_fn=render_composicion,
    extras_widgets=[],
    fuente='CASEN 2017 y 2024 / CENSO 2017 y 2024'
)

## 🖥️ Configuración del Dashboard

In [11]:
# Configuración del Dashboard
 
CONFIGS = [cfg_edu, cfg_viv, cfg_emp, cfg_comp]
tab_widgets, refreshers, dd_años_list, dd_grafs_list = [], [], [], []
 
for cfg in CONFIGS:
    cont, ref, dda, ddg = crear_tab(cfg)
    tab_widgets.append(cont)
    refreshers.append(ref)
    dd_años_list.append(dda)
    dd_grafs_list.append(ddg)
 
tabs = ipw.Tab(children=tab_widgets, layout=ipw.Layout(flex='1'))
for i, cfg in enumerate(CONFIGS):
    tabs.set_title(i, cfg['nombre'])
 
# Checkboxes por comuna
checks = {c: ipw.Checkbox(value=(i < 3), description=c, indent=False,
                           layout=ipw.Layout(width='155px'))
          for i, c in enumerate(COMUNAS_ALL)}
 
btn_ok = ipw.Button(description='Actualizar', layout=ipw.Layout(width='135px', margin='8px 0 0 0'))
btn_sel_all = ipw.Button(description='Todas', layout=ipw.Layout(width='64px'))
btn_sel_none = ipw.Button(description='Ninguna', layout=ipw.Layout(width='64px'))
 
def _sel_all(_):
    for cb in checks.values(): cb.value = True
def _sel_none(_):
    for cb in checks.values(): cb.value = False
 
btn_sel_all.on_click(_sel_all)
btn_sel_none.on_click(_sel_none)
 
panel_izq = ipw.VBox(
    [
        ipw.HTML('<div style="font-size:11px;color:#888;margin-bottom:4px">Comunas:</div>'),
        ipw.HBox([btn_sel_all, btn_sel_none], layout=ipw.Layout(gap='4px', margin='0 0 6px 0')),
        ipw.VBox(list(checks.values()), layout=ipw.Layout(overflow_y='auto', max_height='260px')),
        btn_ok,
    ],
    layout=ipw.Layout(padding='14px 10px 14px 14px', min_width='175px', border_right='1.5px solid #EBEBEB')
)
 
titulo = ipw.HTML("""
<div style="background:#185FA5;color:white;padding:11px 18px;border-radius:8px 8px 0 0">
  <div style="font-size:15px;font-weight:500">Pobreza Multidimensional en la Región de Los Ríos</div>
  <div style="font-size:10px;opacity:0.75;margin-top:2px">Fuente: CASEN 2017 y 2024 · CENSO · ENE · BCN · Portal Inmobiliario</div>
</div>""")
 
cuerpo = ipw.HBox([panel_izq, tabs],
                  layout=ipw.Layout(border='1px solid #E0E0E0', border_radius='0 0 8px 8px',
                                    background='white', min_height='480px'))
dashboard = ipw.VBox([titulo, cuerpo], layout=ipw.Layout(max_width='980px'))

## 🎮 Controladores de Eventos

In [12]:
# Controladores de eventos
 
def _extras_vals(tab_i):
    return (sl_personas.value, sl_dorm.value) if tab_i == 1 else ()
 
def _comunas_sel():
    sel = [c for c, cb in checks.items() if cb.value]
    return sel or COMUNAS_ALL
 
def _refrescar_tab(i):
    comunas = _comunas_sel()
    año, graf, extras = dd_años_list[i].value, dd_grafs_list[i].value, _extras_vals(i)
    refreshers[i](comunas, año, graf, extras)
 
def _refrescar_actual(*_):
    _refrescar_tab(tabs.selected_index)
 
btn_ok.on_click(_refrescar_actual)
tabs.observe(lambda *_: _refrescar_tab(tabs.selected_index), names='selected_index')
 
for i in range(len(CONFIGS)):
    dd_años_list[i].observe(lambda c, idx=i: _refrescar_tab(idx), names='value')
    dd_grafs_list[i].observe(lambda c, idx=i: _refrescar_tab(idx), names='value')
 
sl_personas.observe(lambda c: _refrescar_tab(1), names='value')
sl_dorm.observe(lambda c: _refrescar_tab(1), names='value')

---
## 🤖 Módulo ML — Random Forest y Clustering Jerárquico

## 🔄 Carga de Modelos ML

In [13]:
import joblib

# ── Cargar modelos desde disco ──────────────────────────────────
try:
    rf_idx          = joblib.load("modelos_ml/modelo_rf_idx.pkl")
    INDICES_CLUSTER = joblib.load("modelos_ml/indices_cluster.pkl")
    K               = joblib.load("modelos_ml/k_clusters.pkl")
    df_cl = df_idx  = pd.read_csv("modelos_ml/df_idx.csv")

    # Reconstruir cluster desde los índices si no está en el CSV
    if "cluster" not in df_cl.columns:
        from scipy.cluster.hierarchy import linkage, fcluster
        from sklearn.preprocessing import StandardScaler
        _X = df_cl[INDICES_CLUSTER].dropna()
        _Xs = StandardScaler().fit_transform(_X)
        _Z  = linkage(_Xs, method="ward")
        _lbl = fcluster(_Z, K, criterion="maxclust")
        df_cl["cluster"] = np.nan
        df_cl.loc[_X.index, "cluster"] = _lbl
        print(f"   ⚙️  cluster reconstruido al vuelo (K={K})")
    mask_cluster = df_cl["cluster"].notna()
    labels       = df_cl.loc[mask_cluster, "cluster"].astype(int).values

    ML_OK = True
    print(f"✅ Modelos cargados")
    print(f"   K={K} | indices: {INDICES_CLUSTER}")
    print(f"   filas: {df_idx.shape} | clusters válidos: {len(labels)}")

except FileNotFoundError as e:
    ML_OK = False
    print(f"⚠️  Archivo no encontrado: {e}")
    print("   Ejecuta random_forest_losrios.ipynb completo y corre la ultima celda de guardado.")

COLS_DIM_MAP = {
    "Educacion": "#4e79a7", "Vivienda": "#f28e2b",
    "Composicion": "#59a14f", "Ingresos": "#e15759"
}
COLS_CLUSTER = ["#4e79a7","#f28e2b","#e15759","#59a14f","#b07aa1","#76b7b2"]

   ⚙️  cluster reconstruido al vuelo (K=4)
✅ Modelos cargados
   K=4 | indices: ['idx_educacion', 'idx_vivienda', 'idx_composicion', 'idx_ingresos']
   filas: (6608, 63) | clusters válidos: 6608


## 🌲 Tab: Random Forest — Clasificación Supervisada

In [14]:
# ── TAB RANDOM FOREST ───────────────────────────────────────────

# Sliders predictor
sl_edu  = ipw.FloatSlider(min=0,max=1,step=0.01,value=0.5,description='Educacion:',
    continuous_update=False,style={'description_width':'100px'},layout=ipw.Layout(width='340px'))
sl_viv  = ipw.FloatSlider(min=0,max=1,step=0.01,value=0.5,description='Vivienda:',
    continuous_update=False,style={'description_width':'100px'},layout=ipw.Layout(width='340px'))
sl_comp = ipw.FloatSlider(min=0,max=1,step=0.01,value=0.5,description='Composicion:',
    continuous_update=False,style={'description_width':'100px'},layout=ipw.Layout(width='340px'))
sl_ing  = ipw.FloatSlider(min=0,max=1,step=0.01,value=0.5,description='Ingresos:',
    continuous_update=False,style={'description_width':'100px'},layout=ipw.Layout(width='340px'))

out_pred_html = ipw.HTML()

def predecir_hogar(*args):
    if not ML_OK:
        out_pred_html.value = '<p style="color:#E24B4A">Modelo no disponible</p>'
        return
    vals = np.array([[sl_edu.value, sl_viv.value, sl_comp.value, sl_ing.value]])
    prob = rf_idx.predict_proba(vals)[0][1]
    pred = rf_idx.predict(vals)[0]
    color = '#E24B4A' if pred == 1 else '#1D9E75'
    label = 'POBRE MULTIDIMENSIONAL' if pred == 1 else 'NO POBRE'
    barra = (
        '<div style="background:#eee;border-radius:4px;height:10px;width:200px">'
        '<div style="background:' + color + ';width:' + str(int(prob*100)) + '%;'
        'height:10px;border-radius:4px"></div></div>'
    )
    out_pred_html.value = (
        '<div style="padding:12px;background:#F4F6FA;border-radius:8px;'
        'border-left:4px solid ' + color + ';margin-top:8px;max-width:400px">'
        '<div style="font-size:13px;font-weight:500;color:' + color + '">' + label + '</div>'
        '<div style="font-size:11px;color:#666;margin-top:6px">Probabilidad: <b>' + f'{prob:.1%}' + '</b></div>'
        '<div style="margin-top:6px">' + barra + '</div>'
        '<div style="font-size:10px;color:#999;margin-top:8px">0=sin carencia · 1=maxima carencia</div>'
        '</div>'
    )

for sl in [sl_edu, sl_viv, sl_comp, sl_ing]:
    sl.observe(predecir_hogar, names='value')
predecir_hogar()

# Graficos RF
out_rf_main = ipw.Output()
dd_rf = ipw.Dropdown(
    options=[('Importancia de dimensiones','importancia'),
             ('Predicciones por comuna','comunas'),
             ('Predictor de hogar','predictor')],
    layout=ipw.Layout(width='250px'))

def render_rf(*args):
    with out_rf_main:
        clear_output(wait=True)
        op = dd_rf.value
        if not ML_OK:
            print("Modelo no disponible")
            return

        if op == 'importancia':
            imp = pd.Series(rf_idx.feature_importances_, index=INDICES_CLUSTER)
            imp.index = [i.replace('idx_','').capitalize() for i in imp.index]
            imp = imp.sort_values()
            fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
            fig.patch.set_facecolor(CBG)
            colb = [COLS_DIM_MAP.get(d,'#76b7b2') for d in imp.index]
            bars = axes[0].barh(imp.index, imp.values, color=colb, edgecolor='white', height=0.5)
            axes[0].set_xlabel('Importancia', fontsize=9)
            axes[0].set_title('Importancia por dimension', fontsize=10, fontweight='bold')
            axes[0].grid(axis='x', alpha=0.3)
            axes[0].set_facecolor('white')
            axes[0].spines[['top','right']].set_visible(False)
            for bar, v in zip(bars, imp.values):
                axes[0].text(v+0.005, bar.get_y()+bar.get_height()/2, f'{v:.3f}', va='center', fontsize=9)
            colp = [COLS_DIM_MAP.get(d,'#76b7b2') for d in imp.index]
            axes[1].pie(imp.values, labels=imp.index, colors=colp,
                       autopct='%1.1f%%', startangle=90,
                       wedgeprops={'edgecolor':'white','linewidth':2})
            axes[1].set_title('Distribucion del peso', fontsize=10, fontweight='bold')
            plt.tight_layout()
            plt.show()

        elif op == 'comunas':
            comunas_sel = _comunas_sel()
            if 'comuna' not in df_idx.columns:
                print("Columna 'comuna' no disponible")
                return
            df_p = df_idx[INDICES_CLUSTER + ['pobreza_multi','comuna']].dropna().copy()
            df_p['prediccion'] = rf_idx.predict(df_p[INDICES_CLUSTER])
            df_p = df_p[df_p['comuna'].isin(comunas_sel)]
            res = df_p.groupby('comuna').agg(
                pct_real=('pobreza_multi','mean'),
                pct_pred=('prediccion','mean')
            ).reset_index().sort_values('pct_real', ascending=False)
            res[['pct_real','pct_pred']] *= 100
            fig, ax = plt.subplots(figsize=(9, 3.8))
            fig.patch.set_facecolor(CBG)
            x = np.arange(len(res)); w = 0.35
            ax.bar(x-w/2, res['pct_real'], w, label='Real', color=C17, alpha=0.85)
            ax.bar(x+w/2, res['pct_pred'], w, label='Predicho', color=C24, alpha=0.85)
            ax.set_xticks(x)
            ax.set_xticklabels(res['comuna'], rotation=30, ha='right', fontsize=9)
            ax.set_ylabel('% hogares pobres', fontsize=9)
            ax.set_title('Pobreza real vs predicha por comuna', fontsize=10, fontweight='bold')
            ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
            ax.set_facecolor('white'); ax.spines[['top','right']].set_visible(False)
            plt.tight_layout(); plt.show()

        elif op == 'predictor':
            display(ipw.VBox([
                ipw.HTML('<div style="font-size:11px;color:#888;margin-bottom:6px">'
                         'Ajusta los indices de carencia (0=sin carencia, 1=maxima carencia)</div>'),
                sl_edu, sl_viv, sl_comp, sl_ing, out_pred_html
            ]))

dd_rf.observe(render_rf, names='value')

tab_rf_content = ipw.VBox([
    ipw.HTML('<div style="padding:6px 0 8px;border-bottom:1.5px solid #E8E8E8;margin-bottom:10px">'
             '<span style="font-size:15px;font-weight:500">Random Forest — Clasificacion supervisada</span></div>'),
    ipw.HTML('<div style="font-size:10px;color:#bbb;margin-bottom:8px">Modelo entrenado con CASEN 2017+2024 · Target: pobreza_multi (MDSF)</div>'),
    ipw.HBox([ipw.HTML('<span style="font-size:11px;color:#999;line-height:28px">Vista:</span>'), dd_rf],
             layout=ipw.Layout(align_items='center', gap='5px', margin='0 0 8px 0')),
    out_rf_main
], layout=ipw.Layout(padding='14px'))

render_rf()
print("✅ Tab Random Forest listo")

✅ Tab Random Forest listo


## 🔵 Tab: Clustering Jerárquico — Aprendizaje No Supervisado

In [15]:
# ── TAB CLUSTERING ──────────────────────────────────────────────

out_clust_main = ipw.Output()
dd_clust = ipw.Dropdown(
    options=[('Radar de perfiles','radar'),
             ('Heatmap de carencias','heatmap'),
             ('Clusters por comuna','comunas')],
    layout=ipw.Layout(width='250px'))

def render_clust(*args):
    with out_clust_main:
        clear_output(wait=True)
        if not ML_OK:
            print("Modelo no disponible")
            return
        op = dd_clust.value
        comunas_sel = _comunas_sel()

        # Reconstruir perfil desde df_cl (contiene columna cluster)
        X_c = df_cl[INDICES_CLUSTER].copy()
        mask = X_c.notna().all(axis=1) & df_cl['pobreza_multi'].notna()
        X_c = X_c[mask].reset_index(drop=True)
        meta = df_cl[mask][['pobreza_multi','cluster'] + (['comuna'] if 'comuna' in df_cl.columns else [])].reset_index(drop=True).copy()
        meta['cluster'] = meta['cluster'].astype(int)

        perfil = X_c.copy()
        perfil['cluster'] = meta['cluster'].values
        perfil_mean = perfil.groupby('cluster')[INDICES_CLUSTER].mean()
        perfil_mean.index = [f'Cluster {i}' for i in perfil_mean.index]
        pct_pobres = meta.groupby('cluster')['pobreza_multi'].mean() * 100
        n_hogares  = meta.groupby('cluster').size()
        dims = [c.replace('idx_','').capitalize() for c in INDICES_CLUSTER]

        if op == 'radar':
            angulos = np.linspace(0, 2*np.pi, len(dims), endpoint=False).tolist()
            angulos += angulos[:1]
            fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
            fig.patch.set_facecolor(CBG)
            for i, (idx_cl, row) in enumerate(perfil_mean.iterrows()):
                vals = row.tolist() + row.tolist()[:1]
                color = COLS_CLUSTER[i % len(COLS_CLUSTER)]
                cl_num = int(idx_cl.split()[-1])
                pct = pct_pobres.get(cl_num, 0)
                n   = n_hogares.get(cl_num, 0)
                ax.plot(angulos, vals, 'o-', lw=2, color=color,
                        label=f'{idx_cl} (n={n}, {pct:.0f}% pobres)')
                ax.fill(angulos, vals, alpha=0.12, color=color)
            ax.set_xticks(angulos[:-1])
            ax.set_xticklabels(dims, fontsize=12, fontweight='bold')
            ax.set_ylim(0, 1)
            ax.set_title(f'Perfiles de Cluster (K={K})', fontsize=12, fontweight='bold', pad=20)
            ax.legend(loc='upper right', bbox_to_anchor=(1.4, 1.15), fontsize=9)
            ax.grid(alpha=0.3)
            plt.tight_layout(); plt.show()

        elif op == 'heatmap':
            pm = perfil_mean.copy()
            pm.columns = dims
            fig, ax = plt.subplots(figsize=(8, max(3, K*0.8)))
            fig.patch.set_facecolor(CBG)
            sns.heatmap(pm, annot=True, fmt='.2f', cmap='RdYlGn_r',
                       vmin=0, vmax=1, linewidths=0.5, ax=ax,
                       cbar_kws={'label': 'Indice de carencia (0-1)'})
            ax.set_title(f'Perfil de carencia por cluster (K={K})', fontsize=11, fontweight='bold')
            plt.tight_layout(); plt.show()

        elif op == 'comunas':
            if 'comuna' not in meta.columns:
                print("Columna 'comuna' no disponible")
                return
            sub = meta[meta['comuna'].isin(comunas_sel)]
            tabla = sub.groupby(['comuna','cluster']).size().unstack(fill_value=0)
            tabla.columns = [f'Cluster {c}' for c in tabla.columns]
            tabla_pct = tabla.div(tabla.sum(axis=1), axis=0) * 100
            tabla_pct = tabla_pct.sort_values(tabla_pct.columns[0], ascending=False)
            fig, ax = plt.subplots(figsize=(9, 4))
            fig.patch.set_facecolor(CBG)
            tabla_pct.plot(kind='bar', ax=ax, stacked=True,
                          color=COLS_CLUSTER[:K], edgecolor='white', width=0.7, alpha=0.9)
            ax.set_ylabel('% hogares', fontsize=9)
            ax.set_title(f'Composicion de clusters por comuna (K={K})', fontsize=10, fontweight='bold')
            ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=9)
            ax.set_ylim(0, 100)
            ax.legend(title='Cluster', fontsize=9, bbox_to_anchor=(1.01,1), loc='upper left')
            ax.grid(axis='y', alpha=0.3)
            ax.set_facecolor('white'); ax.spines[['top','right']].set_visible(False)
            plt.tight_layout(); plt.show()

dd_clust.observe(render_clust, names='value')

k_str = str(K) if ML_OK else '?'
tab_clust_content = ipw.VBox([
    ipw.HTML('<div style="padding:6px 0 8px;border-bottom:1.5px solid #E8E8E8;margin-bottom:10px">'
             '<span style="font-size:15px;font-weight:500">Clustering Jerarquico — Aprendizaje no supervisado</span></div>'),
    ipw.HTML('<div style="font-size:10px;color:#bbb;margin-bottom:8px">Metodo Ward · K=' + k_str + ' clusters · Indices IPM 0-1</div>'),
    ipw.HBox([ipw.HTML('<span style="font-size:11px;color:#999;line-height:28px">Vista:</span>'), dd_clust],
             layout=ipw.Layout(align_items='center', gap='5px', margin='0 0 8px 0')),
    out_clust_main
], layout=ipw.Layout(padding='14px'))

render_clust()
print("✅ Tab Clustering listo")

✅ Tab Clustering listo


## 🔗 Reensamblado del Dashboard con Tabs ML

In [16]:
# ── REENSAMBLAR DASHBOARD CON TABS ML ───────────────────────────

tabs_ml = ipw.Tab(
    children=tab_widgets + [tab_rf_content, tab_clust_content],
    layout=ipw.Layout(flex='1')
)
nombres_tabs = [cfg['nombre'] for cfg in CONFIGS] + ['Random Forest', 'Clustering']
for i, nombre in enumerate(nombres_tabs):
    tabs_ml.set_title(i, nombre)

cuerpo_ml = ipw.HBox(
    [panel_izq, tabs_ml],
    layout=ipw.Layout(border='1px solid #E0E0E0', border_radius='0 0 8px 8px',
                      min_height='480px')
)
titulo_ml = ipw.HTML("""
<div style="background:#185FA5;color:white;padding:11px 18px;border-radius:8px 8px 0 0">
  <div style="font-size:15px;font-weight:500">Pobreza Multidimensional — Region de Los Rios</div>
  <div style="font-size:10px;opacity:0.75;margin-top:2px">
    CASEN 2017+2024 · Random Forest · Clustering Jerarquico
  </div>
</div>""")

dashboard_ml = ipw.VBox(
    [titulo_ml, cuerpo_ml],
    layout=ipw.Layout(max_width='1020px')
)

def _refrescar_ml(*_):
    idx = tabs_ml.selected_index
    if idx < len(CONFIGS):
        _refrescar_tab(idx)
    elif idx == len(CONFIGS):
        render_rf()
    else:
        render_clust()

btn_ok.on_click(_refrescar_ml)
tabs_ml.observe(lambda *_: _refrescar_ml(), names='selected_index')

display(dashboard_ml)
_refrescar_ml()